## anamoly detection
- here we will detect the anomaly scenario
- for example: the model will see "This 52,000 education payment is suspicious — 
average education spend is 8,000"

- we have two approaches for this, we will implement both and use the best one

- approach 1 — Statistical (Z-Score)
Flag any transaction where amount is more than 2 standard deviations above the category mean. Simple, explainable, fast.
- approach 2 — Isolation Forest
A proper ML algorithm designed specifically for anomaly detection. More sophisticated, handles complex patterns.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

In [3]:
df = pd.read_excel("../data/sample_transactions.xlsx")
df.head()

,transaction_id,date,day_of_week,month,description,merchant,amount,category,payment_mode,transaction_channel,account_type,city,is_weekend,is_anomaly
0,TXN00001,2024-01-02,Tuesday,January,POS TXN 9289,Cafe Coffee Day,8123.00,Food,Cash,ATM,Savings,Delhi,False,True
1,TXN00002,2024-01-02,Tuesday,January,dunzo delivery,Dunzo,315.93,Food,NEFT,Web,Savings,Delhi,False,False
2,TXN00003,2024-01-02,Tuesday,January,property tax payment,BBMP,4619.46,Utilities,Debit Card,Auto Debit,Savings,Delhi,False,False
3,TXN00004,2024-01-02,Tuesday,January,monthly rent,Housing Society,14768.18,Rent,Net Banking,Auto Debit,Savings,Delhi,False,False
4,TXN00005,2024-01-02,Tuesday,January,monthly rent,Housing Society,20437.67,Rent,Auto Debit,Auto Debit,Savings,Delhi,False,False


In [4]:
## converting date
df['date'] = pd.to_datetime(df['date'])
print(df['date'].dtype)

datetime64[us]


In [5]:
## cleaning description text
df['description']= df['description'].str.lower().str.strip()

### Approach 1: Z-score

- for each category we will find 'mean' and 'standard deviation'
- the for each transaction z score is computed
   - z-score = (amount - mean)/std
   - if z-score > 2, then we flag it as an anomaly

In [6]:
## implementing approach 1

category_stats = df.groupby('category')['amount'].agg(['mean','std']).reset_index()
category_stats.columns=['category', 'mean_amount', 'std_amount']
print(category_stats)

         category   mean_amount    std_amount
0           Bills   2467.164545   2200.135946
1       Education  17838.963617  16194.819708
2   Entertainment   1499.183960   1145.203114
3         Fitness   4735.143750   4147.539363
4            Food    927.652878   1560.256312
5            Fuel   1985.153934   1495.235392
6       Groceries   2733.426923   2556.733620
7      Healthcare   4227.100656   2385.475138
8       Insurance   8204.649118   4917.545191
9     Investments  15573.175082   8982.825886
10  Personal Care   2853.718000   1675.515805
11           Rent  15474.642533   5877.428922
12       Shopping   5871.068387   5463.501740
13         Travel   2912.124257   3623.276553
14      Utilities   3533.739595   2624.102296


In [7]:
## merging back into the maine dataframe

df = df.merge(category_stats, on='category')

# on decides based on which column both the dataframes are being merged

df.head()

,transaction_id,date,day_of_week,month,description,merchant,amount,category,payment_mode,transaction_channel,account_type,city,is_weekend,is_anomaly,mean_amount,std_amount
0,TXN00001,2024-01-02,Tuesday,January,pos txn 9289,Cafe Coffee Day,8123.00,Food,Cash,ATM,Savings,Delhi,False,True,927.652878,1560.256312
1,TXN00002,2024-01-02,Tuesday,January,dunzo delivery,Dunzo,315.93,Food,NEFT,Web,Savings,Delhi,False,False,927.652878,1560.256312
2,TXN00003,2024-01-02,Tuesday,January,property tax payment,BBMP,4619.46,Utilities,Debit Card,Auto Debit,Savings,Delhi,False,False,3533.739595,2624.102296
3,TXN00004,2024-01-02,Tuesday,January,monthly rent,Housing Society,14768.18,Rent,Net Banking,Auto Debit,Savings,Delhi,False,False,15474.642533,5877.428922
4,TXN00005,2024-01-02,Tuesday,January,monthly rent,Housing Society,20437.67,Rent,Auto Debit,Auto Debit,Savings,Delhi,False,False,15474.642533,5877.428922


In [8]:
## computing the z_score
df['z_score'] = (df['amount']-df['mean_amount']) / df['std_amount']

In [27]:
# flag anamonaly column
df['anomaly_zscore'] = df['z_score']>2.0

df.head()

,transaction_id,date,day_of_week,month,description,merchant,amount,category,payment_mode,transaction_channel,account_type,city,is_weekend,is_anomaly,mean_amount,std_amount,z_score,anomaly_zscore,category_encoded,anomaly_iforest
0,TXN00001,2024-01-02,Tuesday,January,pos txn 9289,Cafe Coffee Day,8123.00,Food,Cash,ATM,Savings,Delhi,False,True,927.652878,1560.256312,4.611644,True,4,False
1,TXN00002,2024-01-02,Tuesday,January,dunzo delivery,Dunzo,315.93,Food,NEFT,Web,Savings,Delhi,False,False,927.652878,1560.256312,-0.392066,False,4,False
2,TXN00003,2024-01-02,Tuesday,January,property tax payment,BBMP,4619.46,Utilities,Debit Card,Auto Debit,Savings,Delhi,False,False,3533.739595,2624.102296,0.413749,False,14,False
3,TXN00004,2024-01-02,Tuesday,January,monthly rent,Housing Society,14768.18,Rent,Net Banking,Auto Debit,Savings,Delhi,False,False,15474.642533,5877.428922,-0.120199,False,11,False
4,TXN00005,2024-01-02,Tuesday,January,monthly rent,Housing Society,20437.67,Rent,Auto Debit,Auto Debit,Savings,Delhi,False,False,15474.642533,5877.428922,0.844422,False,11,False


In [28]:
## now we check how many rows got anomaly detection
print("number of actual anomalies in data: ", df['is_anomaly'].sum())
print("Anomalies detected: ",df['anomaly_zscore'].sum())

print(df[df['anomaly_zscore']==True][['description', 'category', 'amount', 'z_score']])

number of actual anomalies in data:  55
Anomalies detected:  38
                 description       category    amount   z_score
0               pos txn 9289           Food   8123.00  4.611644
14                 zmto*6301           Food  11294.10  6.644067
29    online course purchase      Education  81218.00  3.913538
39           monthly grocery      Groceries  18348.40  6.107391
60                   bb*6315      Groceries  17383.40  5.729957
69       beauty subscription  Personal Care   9321.12  3.859947
118         account transfer    Investments  41858.88  2.926218
130            wellness plan        Fitness  14288.24  2.303317
151          bigbasket order      Groceries  11619.40  3.475518
302     subscription renewal      Utilities  18632.60  5.753915
318               nykaa*7445       Shopping  37475.15  5.784583
322         late night order           Food   6448.60  3.538487
331             neft payment         Travel  30550.52  7.628012
395     auto debit insurance      Insura

## Approach 2: isolation forest
it looks at the entire dataset and finds points that are easisest to isolate from the rest

- two features will be given as input: amount and category (in encoded form)

In [29]:
from sklearn.preprocessing import StandardScaler

#encoding category as numbers
df['category_encoded'] = df['category'].astype('category').cat.codes

# prepareing the features
features = df[['amount', 'category_encoded']]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [30]:
# training isolation forest

iso_forest = IsolationForest(contamination=0.04, random_state=42)
df['anomaly_iforest'] = iso_forest.fit_predict(features_scaled)

# we need iso forest to return -1 for anomaly and 1 for normal

df['anomaly_iforest'] = df['anomaly_iforest'] == -1

##checking the total number of anomalies detected
print('anomalies detected by isolation forest',df['anomaly_iforest'].sum())

df.head(2)

anomalies detected by isolation forest 56


,transaction_id,date,day_of_week,month,description,merchant,amount,category,payment_mode,transaction_channel,account_type,city,is_weekend,is_anomaly,mean_amount,std_amount,z_score,anomaly_zscore,category_encoded,anomaly_iforest
0,TXN00001,2024-01-02,Tuesday,January,pos txn 9289,Cafe Coffee Day,8123.00,Food,Cash,ATM,Savings,Delhi,False,True,927.652878,1560.256312,4.611644,True,4,False
1,TXN00002,2024-01-02,Tuesday,January,dunzo delivery,Dunzo,315.93,Food,NEFT,Web,Savings,Delhi,False,False,927.652878,1560.256312,-0.392066,False,4,False


In [31]:
## checking the classification report of both z score and isolation forest

from sklearn.metrics import classification_report, accuracy_score

## for z score
print("z-score")
print(accuracy_score(df["is_anomaly"], df['anomaly_zscore']))
print(classification_report(df["is_anomaly"], df['anomaly_zscore']))

## for isolation forest
print("isolation forest")
print(accuracy_score(df["is_anomaly"], df['anomaly_iforest']))
print(classification_report(df["is_anomaly"], df['anomaly_iforest']))

z-score
0.9878397711015737
              precision    recall  f1-score   support

       False       0.99      1.00      0.99      1343
        True       1.00      0.69      0.82        55

    accuracy                           0.99      1398
   macro avg       0.99      0.85      0.91      1398
weighted avg       0.99      0.99      0.99      1398

isolation forest
0.9506437768240343
              precision    recall  f1-score   support

       False       0.97      0.97      0.97      1343
        True       0.38      0.38      0.38        55

    accuracy                           0.95      1398
   macro avg       0.67      0.68      0.68      1398
weighted avg       0.95      0.95      0.95      1398



Z-Score worked better than Isolation Forest because it sets thresholds within each category, which makes more sense when spending habits differ a lot across categories

- Z-Score: Precision 1.0, Recall 0.69, F1 0.82
- Isolation Forest: Precision 0.38, Recall 0.38, F1 0.38

In [32]:
# we wil be using z score with threhold value of 2.0
ANOMALY_THRESHOLD = 2.0

In [33]:
## we do not need a trained model file to be saved for Z-score
#we will just save the anomaly threshold we are using
## and the columns used in computing z scorw which are saved in category_stats

joblib.dump(ANOMALY_THRESHOLD, '../models/anomaly_threshold.pkl')
print("Anomaaly threshold saved")

joblib.dump(category_stats,'../models/anomaly_category_stats.pkl')
print("category stats saved")

Anomaaly threshold saved
category stats saved
